# StructEval Evaluation Notebook

このノートブックは、推論結果JSONを読み込み、評価を行います。

## 評価の構成
- **ミクロ評価**: 各問題ごとの正答状況を表示
- **マクロ評価**: カテゴリ別・全体の総合点を表示

## 1. セットアップ

In [ ]:
import json
import os
import sys
import pandas as pd
from typing import Dict, List, Any, Optional
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Add structeval to path
sys.path.insert(0, os.path.join(os.getcwd(), "structeval"))

print("Setup complete!")

## 2. 設定パラメータ

評価に必要なパラメータを設定します。

In [ ]:
# ============================================
# 設定パラメータ（必要に応じて変更してください）
# ============================================

# 推論結果JSONのパス
INFERENCE_RESULTS_PATH = "output/inference_results.json"

# レンダリング出力先（レンダリングを実行する場合）
IMG_OUTPUT_PATH = "output/rendered_images"
NON_RENDERABLE_OUTPUT_DIR = "output/non_renderable_files"

# VLMモデル設定（VQA評価を行う場合）
VLM_MODEL_NAME = "gpt-4o"  # VLMモデル名
VLM_ENGINE = "openai"  # 推論エンジン

# 評価結果の保存先
EVALUATION_OUTPUT_PATH = "output/evaluation_results.json"

print("Configuration loaded.")
print(f"  Inference results: {INFERENCE_RESULTS_PATH}")
print(f"  Image output: {IMG_OUTPUT_PATH}")
print(f"  VLM model: {VLM_MODEL_NAME} ({VLM_ENGINE})")

## 3. 推論結果の読み込み

In [ ]:
def load_inference_results(path: str) -> List[Dict[str, Any]]:
    """Load inference results from JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Load the data
inference_data = load_inference_results(INFERENCE_RESULTS_PATH)

print(f"Loaded {len(inference_data)} tasks from inference results.")

# Show sample
if inference_data:
    sample = inference_data[0]
    print(f"\nSample task:")
    print(f"  Task ID: {sample.get('task_id')}")
    print(f"  Task Name: {sample.get('task_name')}")
    print(f"  Input Type: {sample.get('input_type')}")
    print(f"  Output Type: {sample.get('output_type')}")
    print(f"  Renderable: {sample.get('rendering')}")
    print(f"  Generation length: {len(sample.get('generation', ''))} chars")

## 4. データ概要の確認

In [ ]:
def analyze_dataset(data: List[Dict[str, Any]]) -> pd.DataFrame:
    """Analyze the dataset and return summary statistics."""
    summary = {
        "Total Tasks": len(data),
        "Renderable Tasks": sum(1 for d in data if d.get("rendering", False)),
        "Non-Renderable Tasks": sum(1 for d in data if not d.get("rendering", False)),
    }
    
    # Count by input type
    input_types = {}
    for item in data:
        input_type = item.get("input_type", "Unknown")
        input_types[input_type] = input_types.get(input_type, 0) + 1
    
    # Count by output type
    output_types = {}
    for item in data:
        output_type = item.get("output_type", "Unknown")
        output_types[output_type] = output_types.get(output_type, 0) + 1
    
    return summary, input_types, output_types

summary, input_types, output_types = analyze_dataset(inference_data)

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n--- Input Types ---")
for itype, count in sorted(input_types.items()):
    print(f"  {itype}: {count}")

print("\n--- Output Types ---")
for otype, count in sorted(output_types.items()):
    print(f"  {otype}: {count}")

## 5. レンダリング処理（オプション）

生成されたコードをレンダリングして画像を生成します。VQA評価を行う場合は必須です。

In [ ]:
# Set to True if you want to run rendering
RUN_RENDERING = False

if RUN_RENDERING:
    import asyncio
    from render_engine.main import process_json_file as render_task
    
    # Create output directories
    os.makedirs(IMG_OUTPUT_PATH, exist_ok=True)
    os.makedirs(NON_RENDERABLE_OUTPUT_DIR, exist_ok=True)
    
    # Run rendering
    print("Starting rendering process...")
    await render_task(INFERENCE_RESULTS_PATH, IMG_OUTPUT_PATH, NON_RENDERABLE_OUTPUT_DIR)
    print("Rendering complete!")
else:
    print("Skipping rendering. Set RUN_RENDERING = True to enable.")

## 6. 評価の実行

In [ ]:
from eval_engine.main import evaluate_dataset, calculate_final_score
from eval_engine.eval_utils import raw_output_eval

def run_evaluation(
    data: List[Dict[str, Any]],
    img_path: str,
    non_renderable_dir: str,
    vlm_model_name: Optional[str] = None,
    vlm_engine: Optional[str] = None,
    **kwargs
) -> List[Dict[str, Any]]:
    """Run evaluation on the inference results."""
    
    # Build image mapping
    images = {}
    for item in data:
        task_id = item.get("task_id")
        img_file = os.path.join(img_path, f"{task_id}.png")
        if os.path.exists(img_file):
            images[task_id] = img_file
    
    print(f"Found {len(images)} rendered images for evaluation.")
    
    # Run evaluation
    results = evaluate_dataset(
        data,
        images,
        vlm_model_name=vlm_model_name,
        vlm_engine=vlm_engine,
        non_renderable_dir=non_renderable_dir,
        **kwargs
    )
    
    return results

# Run evaluation
print("Starting evaluation...")
evaluation_results = run_evaluation(
    inference_data,
    IMG_OUTPUT_PATH,
    NON_RENDERABLE_OUTPUT_DIR,
    vlm_model_name=VLM_MODEL_NAME,
    vlm_engine=VLM_ENGINE
)

print(f"\nEvaluation complete! {len(evaluation_results)} tasks evaluated.")

## 7. ミクロ評価（各問題の正答状況）

各タスクごとの詳細な評価結果を表示します。

In [ ]:
def create_micro_evaluation_df(results: List[Dict[str, Any]]) -> pd.DataFrame:
    """Create a DataFrame with micro-level evaluation results."""
    rows = []
    for item in results:
        row = {
            "Task ID": item.get("task_id", ""),
            "Task Name": item.get("task_name", ""),
            "Input Type": item.get("input_type", ""),
            "Output Type": item.get("output_type", ""),
            "Renderable": "Yes" if item.get("rendering", False) else "No",
            "Render Score": item.get("render_score", 0),
            "Raw Output Score": item.get("raw_output_score", None),
            "VQA Score": item.get("VQA_score", None),
            "Key Validation Score": item.get("key_validation_score", None),
            "Final Score": item.get("final_eval_score", 0)
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

micro_df = create_micro_evaluation_df(evaluation_results)

print("=" * 70)
print("MICRO EVALUATION - Individual Task Results")
print("=" * 70)
display(micro_df)

In [ ]:
# Filter and display specific categories

print("\n--- High Scoring Tasks (Final Score >= 0.8) ---")
high_score_df = micro_df[micro_df["Final Score"] >= 0.8]
print(f"Count: {len(high_score_df)}")
display(high_score_df)

print("\n--- Low Scoring Tasks (Final Score < 0.5) ---")
low_score_df = micro_df[micro_df["Final Score"] < 0.5]
print(f"Count: {len(low_score_df)}")
display(low_score_df)

In [ ]:
# Detailed view for a specific task

def show_task_detail(task_id: str, results: List[Dict[str, Any]]):
    """Show detailed evaluation for a specific task."""
    task = next((r for r in results if r.get("task_id") == task_id), None)
    if not task:
        print(f"Task {task_id} not found.")
        return
    
    print("=" * 70)
    print(f"TASK DETAIL: {task_id}")
    print("=" * 70)
    print(f"Task Name: {task.get('task_name')}")
    print(f"Input Type: {task.get('input_type')}")
    print(f"Output Type: {task.get('output_type')}")
    print(f"Renderable: {task.get('rendering')}")
    print()
    
    print("--- Scores ---")
    print(f"Render Score: {task.get('render_score', 'N/A')}")
    print(f"Raw Output Score: {task.get('raw_output_score', 'N/A')}")
    print(f"VQA Score: {task.get('VQA_score', 'N/A')}")
    print(f"Key Validation Score: {task.get('key_validation_score', 'N/A')}")
    print(f"Final Score: {task.get('final_eval_score', 'N/A')}")
    print()
    
    # Raw output evaluation details
    if task.get("raw_output_eval"):
        print("--- Raw Output Metric Evaluation ---")
        raw_metrics = task.get("raw_output_metric", [])
        raw_eval = task.get("raw_output_eval", [])
        for i, (metric, result) in enumerate(zip(raw_metrics, raw_eval)):
            status = "✓" if result == "True" or result == True else "✗"
            print(f"  {status} {metric}")
    
    # VQA evaluation details
    if task.get("VQAeval"):
        print("\n--- VQA Evaluation ---")
        vqa_pairs = task.get("VQA", [])
        vqa_eval = task.get("VQAeval", [])
        for i, (pair, result) in enumerate(zip(vqa_pairs, vqa_eval)):
            status = "✓" if result == True else "✗" if result == False else "?"
            print(f"  {status} Q: {pair.get('question', '')}")
            print(f"      A: {pair.get('answer', '')}")

# Example: show detail for first task
if evaluation_results:
    show_task_detail(evaluation_results[0]["task_id"], evaluation_results)

## 8. マクロ評価（総合点）

カテゴリ別および全体の評価スコアを集計します。

In [ ]:
def calculate_macro_scores(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Calculate macro-level evaluation scores."""
    
    # Overall statistics
    total_tasks = len(results)
    avg_final_score = sum(r.get("final_eval_score", 0) for r in results) / total_tasks if total_tasks > 0 else 0
    
    # Renderable vs Non-renderable
    renderable = [r for r in results if r.get("rendering", False)]
    non_renderable = [r for r in results if not r.get("rendering", False)]
    
    avg_renderable = sum(r.get("final_eval_score", 0) for r in renderable) / len(renderable) if renderable else 0
    avg_non_renderable = sum(r.get("final_eval_score", 0) for r in non_renderable) / len(non_renderable) if non_renderable else 0
    
    # By input type
    by_input_type = {}
    for r in results:
        input_type = r.get("input_type", "Unknown")
        if input_type not in by_input_type:
            by_input_type[input_type] = []
        by_input_type[input_type].append(r.get("final_eval_score", 0))
    
    input_type_scores = {
        k: {"count": len(v), "avg_score": sum(v) / len(v)} 
        for k, v in by_input_type.items()
    }
    
    # By output type
    by_output_type = {}
    for r in results:
        output_type = r.get("output_type", "Unknown")
        if output_type not in by_output_type:
            by_output_type[output_type] = []
        by_output_type[output_type].append(r.get("final_eval_score", 0))
    
    output_type_scores = {
        k: {"count": len(v), "avg_score": sum(v) / len(v)} 
        for k, v in by_output_type.items()
    }
    
    # Component scores
    render_scores = [r.get("render_score", 0) for r in results]
    raw_output_scores = [r.get("raw_output_score", 0) for r in results if r.get("raw_output_score") is not None]
    vqa_scores = [r.get("VQA_score", 0) for r in results if r.get("VQA_score") is not None]
    key_validation_scores = [r.get("key_validation_score", 0) for r in results if r.get("key_validation_score") is not None]
    
    return {
        "overall": {
            "total_tasks": total_tasks,
            "avg_final_score": avg_final_score,
        },
        "by_category": {
            "renderable": {"count": len(renderable), "avg_score": avg_renderable},
            "non_renderable": {"count": len(non_renderable), "avg_score": avg_non_renderable}
        },
        "by_input_type": input_type_scores,
        "by_output_type": output_type_scores,
        "component_scores": {
            "avg_render_score": sum(render_scores) / len(render_scores) if render_scores else 0,
            "avg_raw_output_score": sum(raw_output_scores) / len(raw_output_scores) if raw_output_scores else 0,
            "avg_vqa_score": sum(vqa_scores) / len(vqa_scores) if vqa_scores else 0,
            "avg_key_validation_score": sum(key_validation_scores) / len(key_validation_scores) if key_validation_scores else 0
        }
    }

macro_scores = calculate_macro_scores(evaluation_results)

print("=" * 70)
print("MACRO EVALUATION - Overall Scores")
print("=" * 70)

In [ ]:
# Display overall scores
print("\n" + "=" * 50)
print("OVERALL PERFORMANCE")
print("=" * 50)
print(f"Total Tasks: {macro_scores['overall']['total_tasks']}")
print(f"Average Final Score: {macro_scores['overall']['avg_final_score']:.4f}")

print("\n" + "-" * 50)
print("COMPONENT SCORES")
print("-" * 50)
for comp, score in macro_scores['component_scores'].items():
    print(f"{comp.replace('avg_', '').replace('_', ' ').title()}: {score:.4f}")

In [ ]:
# Display by category
print("\n" + "=" * 50)
print("SCORES BY CATEGORY")
print("=" * 50)

print("\n--- Renderable vs Non-Renderable ---")
for cat, data in macro_scores['by_category'].items():
    print(f"{cat.replace('_', ' ').title()}: {data['count']} tasks, Avg Score: {data['avg_score']:.4f}")

In [ ]:
# Display by input type
print("\n" + "=" * 50)
print("SCORES BY INPUT TYPE")
print("=" * 50)

input_type_df = pd.DataFrame([
    {"Input Type": k, "Count": v["count"], "Avg Score": round(v["avg_score"], 4)}
    for k, v in macro_scores['by_input_type'].items()
]).sort_values("Avg Score", ascending=False)

display(input_type_df)

In [ ]:
# Display by output type
print("\n" + "=" * 50)
print("SCORES BY OUTPUT TYPE")
print("=" * 50)

output_type_df = pd.DataFrame([
    {"Output Type": k, "Count": v["count"], "Avg Score": round(v["avg_score"], 4)}
    for k, v in macro_scores['by_output_type'].items()
]).sort_values("Avg Score", ascending=False)

display(output_type_df)

## 9. 評価結果の可視化

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Score distribution histogram
    ax1 = axes[0, 0]
    scores = [r.get("final_eval_score", 0) for r in evaluation_results]
    ax1.hist(scores, bins=20, edgecolor='black', alpha=0.7)
    ax1.set_xlabel('Final Score')
    ax1.set_ylabel('Count')
    ax1.set_title('Distribution of Final Scores')
    ax1.axvline(x=np.mean(scores), color='red', linestyle='--', label=f'Mean: {np.mean(scores):.3f}')
    ax1.legend()
    
    # 2. Scores by output type
    ax2 = axes[0, 1]
    output_types = list(macro_scores['by_output_type'].keys())
    output_scores = [macro_scores['by_output_type'][ot]['avg_score'] for ot in output_types]
    bars = ax2.barh(output_types, output_scores, color='steelblue', alpha=0.7)
    ax2.set_xlabel('Average Score')
    ax2.set_title('Average Score by Output Type')
    ax2.set_xlim(0, 1)
    
    # 3. Component score comparison
    ax3 = axes[1, 0]
    components = ['Render', 'Raw Output', 'VQA', 'Key Validation']
    comp_scores = [
        macro_scores['component_scores']['avg_render_score'],
        macro_scores['component_scores']['avg_raw_output_score'],
        macro_scores['component_scores']['avg_vqa_score'],
        macro_scores['component_scores']['avg_key_validation_score']
    ]
    colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
    ax3.bar(components, comp_scores, color=colors, alpha=0.7)
    ax3.set_ylabel('Average Score')
    ax3.set_title('Component Scores')
    ax3.set_ylim(0, 1)
    for i, (comp, score) in enumerate(zip(components, comp_scores)):
        ax3.text(i, score + 0.02, f'{score:.3f}', ha='center')
    
    # 4. Renderable vs Non-renderable pie chart
    ax4 = axes[1, 1]
    categories = ['Renderable', 'Non-Renderable']
    counts = [
        macro_scores['by_category']['renderable']['count'],
        macro_scores['by_category']['non_renderable']['count']
    ]
    avg_scores = [
        macro_scores['by_category']['renderable']['avg_score'],
        macro_scores['by_category']['non_renderable']['avg_score']
    ]
    colors = ['#3498db', '#e74c3c']
    wedges, texts, autotexts = ax4.pie(
        counts, 
        labels=[f'{cat}\n({count})' for cat, count in zip(categories, counts)],
        autopct='%1.1f%%',
        colors=colors,
        alpha=0.7
    )
    ax4.set_title(f'Task Distribution\n(Renderable Avg: {avg_scores[0]:.3f}, Non-Renderable Avg: {avg_scores[1]:.3f})')
    
    plt.tight_layout()
    plt.savefig('evaluation_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nVisualization saved to: evaluation_summary.png")
    
except ImportError:
    print("matplotlib not available. Skipping visualization.")

## 10. 評価結果の保存

In [ ]:
def save_evaluation_results(
    results: List[Dict[str, Any]], 
    macro_scores: Dict[str, Any],
    output_path: str
):
    """Save evaluation results to JSON file."""
    
    output = {
        "macro_scores": macro_scores,
        "micro_results": results
    }
    
    # Create output directory if needed
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"Evaluation results saved to: {output_path}")

# Save results
save_evaluation_results(evaluation_results, macro_scores, EVALUATION_OUTPUT_PATH)

In [ ]:
# Also save micro results as CSV for easy viewing
csv_output_path = EVALUATION_OUTPUT_PATH.replace(".json", "_micro.csv")
micro_df.to_csv(csv_output_path, index=False)
print(f"Micro evaluation CSV saved to: {csv_output_path}")

## 11. 評価サマリー

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("EVALUATION COMPLETE - FINAL SUMMARY")
print("=" * 70)
print(f"")
print(f"Total Tasks Evaluated: {macro_scores['overall']['total_tasks']}")
print(f"")
print(f"Overall Average Score: {macro_scores['overall']['avg_final_score']:.4f}")
print(f"")
print(f"By Category:")
print(f"  - Renderable: {macro_scores['by_category']['renderable']['avg_score']:.4f} ({macro_scores['by_category']['renderable']['count']} tasks)")
print(f"  - Non-Renderable: {macro_scores['by_category']['non_renderable']['avg_score']:.4f} ({macro_scores['by_category']['non_renderable']['count']} tasks)")
print(f"")
print(f"Best Performing Output Type: {max(macro_scores['by_output_type'].items(), key=lambda x: x[1]['avg_score'])[0]}")
print(f"Worst Performing Output Type: {min(macro_scores['by_output_type'].items(), key=lambda x: x[1]['avg_score'])[0]}")
print(f"")
print("=" * 70)